# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset. It uses `import json2vec as j2v`, `j2v.Address(...)`, and typed request objects instead of raw field dictionaries.

In [1]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
@j2v.shim(yields=True)
def hello_world_records(_observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
        {"color": "red", "label": "warm"},
        {"color": "blue", "label": "cool"},
    ]

    yield from records

In [3]:
params = j2v.Hyperparameters(
    d_model=16,
    target=[j2v.Address("record", "label")],
    embed=j2v.Address("record"),
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
)


model = j2v.JSON2Vec(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-17 18:45:16.891 | INFO     | json2vec.architecture.root:__init__:128 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(accelerator="cpu", max_epochs=20, logger=False)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃ In sizes ┃ Out sizes ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ 0 │ nodes │ ModuleDict │ 14.5 K │ train │     0 │        ? │         ? │
└───┴───────┴────────────┴────────┴───────┴───────┴──────────┴───────────┘

Trainable params: 14.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 14.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 99                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 75.5 K

/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-05-17 18:45:16.937 | INFO     | json2vec.data.datasets:dataloader:670 - building dataloader
2026-05-17 18:45:16.938 | INFO     | json2vec.data.datasets:__init__:614 - initialized batch dataset


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_c
onnector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the
value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.

2026-05-17 18:45:16.955 | INFO     | json2vec.data.datasets:dataloader:670 - building dataloader
2026-05-17 18:45:16.955 | INFO     | json2vec.data.datasets:__init__:614 - initialized batch dataset


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_c
onnector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=20` reached.


In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [[0.9970940947532654], [0.9972959160804749]],
│   │   │   'null': [[0.00043503890628926456], [0.0007979506044648588]],
│   │   │   'padded': [[0.0006265653646551073], [0.0005605480400845408]],
│   │   │   'masked': [[0.00046465988270938396], [0.000562676228582859]],
│   │   │   'other': [[0.0013796405401080847], [0.0007827258086763322]]
│   │   },
│   │   'content': {
│   │   │   'value': [['warm'], ['cool']],
│   │   │   'probability': [[0.9331893920898438], [0.9863219261169434]],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   [
│   │   │   │   │   │   {'label': 'warm', 'probability': 0.9331893920898438},
│   │   │   │   │   │   {'label': 'cool', 'probability': 0.06681068241596222}
│   │   │   │   │   ]
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   [
│   │   │   │   │   │   {'label': 'cool', 'probability': 0.9863219261169434},
│   │   │   │   │   │   {'label': 'warm', 'probability': 0.01367809996008873}
│   │   │   │   │   ]
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   [
│   │   │   │   │   0.27259504795074463,
│   │   │   │   │   -0.20793849229812622,
│   │   │   │   │   0.3595438599586487,
│   │   │   │   │   0.0287461057305336,
│   │   │   │   │   -0.21888785064220428,
│   │   │   │   │   -0.17636004090309143,
│   │   │   │   │   -0.16742615401744843,
│   │   │   │   │   0.0996888056397438,
│   │   │   │   │   0.042972855269908905,
│   │   │   │   │   -0.1988140195608139,
│   │   │   │   │   -0.1901305764913559,
│   │   │   │   │   -0.08585867285728455,
│   │   │   │   │   0.5072348713874817,
│   │   │   │   │   0.07894451171159744,
│   │   │   │   │   0.340841144323349,
│   │   │   │   │   -0.41326040029525757
│   │   │   │   ]
│   │   │   ],
│   │   │   [
│   │   │   │   [
│   │   │   │   │   -0.1583131104707718,
│   │   │   │   │   0.27216818928718567,
│   │   │   │   │   0.1521625965833664,
│   │   │   │   │   0.29146456718444824,
│   │   │   │   │   0.06437662988901138,
│   │   │   │   │   -0.07499507069587708,
│   │   │   │   │   0.35744574666023254,
│   │   │   │   │   0.05035829171538353,
│   │   │   │   │   -0.024189524352550507,
│   │   │   │   │   0.25411006808280945,
│   │   │   │   │   -0.35584133863449097,
│   │   │   │   │   -0.11330094933509827,
│   │   │   │   │   -0.5488778352737427,
│   │   │   │   │   -0.0827031061053276,
│   │   │   │   │   0.2076740562915802,
│   │   │   │   │   -0.3111782371997833
│   │   │   │   ]
│   │   │   ]
│   │   ]
│   }
}